# AlphaFold in miniature: from a distance map to a 3D fold

*biophysics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/biophysics/mini_alphafold)

A toy version of the two halves of a structure predictor, not AlphaFold itself. A predicted distogram (pairwise Cα distances with binned uncertainty) is optionally cleaned up by one self-attention block, then a "structure module" finds 3D coordinates $x_{ia}$ that fit it, by gradient descent on the stress from a batch of starts and by classical MDS. The result is superposed on the target with the Kabsch SVD and scored by RMSD and a TM-like score.

**Vector form**

$$S(\mathbf X) = \sum_{i<j} w_{ij}\big(\lVert\mathbf x_i - \mathbf x_j\rVert - \hat d_{ij}\big)^2 + k_b\sum_i\big(\lVert\mathbf x_{i+1} - \mathbf x_i\rVert - 3.8\,\text{Å}\big)^2,\qquad \mathbf G = -\tfrac12\,\mathbf J\,\mathbf D^{\circ 2}\,\mathbf J = \mathbf X\mathbf X^{\mathsf T}$$

**Index (tensor) form**

$$\hat d_{ij} = p_{ijk}\,c_k \qquad A_{ij} = \operatorname{softmax}_j\!\Big(\frac{q_{ie}k_{je}}{\sqrt E} + b_{ij}\Big),\quad h'_{ie} = h_{ie} + A_{ij}v_{je} \qquad r_{ij} = \sqrt{(x_{ia}-x_{ja})(x_{ia}-x_{ja})} \qquad H_{ab} = p_{ia}\,q_{ib} = U\Sigma V^{\mathsf T}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{ia}$ | `target_structure(n_helix)` | (N, 3) | Cα position of residue $i$, axis $a$; the target is a synthetic helix–turn–helix built from helix geometry |
| $p_{ijk}$ | `distogram(d_noisy, sigma, centres)` | (N, N, K) | probability that residues $i, j$ sit in distance bin $k$ (64 bins, 2–22 Å); stands in for a trained network |
| $\hat d_{ij} = p_{ijk}c_k$ | `einsum('ijk,k->ij', p, centres)` | (N, N) | expected distance, $k$ summed; the inverse variance gives the confidence weight $w_{ij}$ |
| $q_{ie}k_{je}$ | `einsum('ie,je->ij', q, k)` | (N, N) | attention logits: residue $i$ asks, residue $j$ answers; $e$ summed |
| $A_{ij}v_{je}$ | `einsum('ij,je->ie', A, v)` | (N, E) | each residue collects a weighted mix of the others, $j$ summed; $b_{ij}$ is a pair bias from $\hat d_{ij}$ |
| $r_{ij}$ | `pair_distances(x)` | (B, N, N) | model distances for a batch of $B$ starts, $a$ summed |
| $S$ | `stress(x, d, w, far, d_far, k_bond)` | (B,) | weighted misfit, "far" pairs only pushed apart, bonds held at 3.8 Å; minimised with Adam via autograd |
| $G_{ij}$ | `classical_mds(D)` | (N, 3) | double-centred squared distances = Gram matrix $x_{ia}x_{ja}$; its top 3 eigenvectors give an initial guess |
| $H_{ab}$ | `kabsch(P, Q)` | (N, 3) | covariance of the two centred structures, $i$ summed; its SVD gives the best proper rotation |
| $\text{RMSD},\ \text{TM}$ | `rmsd(P, Q), tm_score(P, Q)` | () | error after superposition; TM-like score $\frac1N\sum_i 1/(1+(d_i/d_0)^2)$, 1 = identical |

This is AlphaFold in spirit only. Real AlphaFold 2 reads the amino-acid sequence and a multiple sequence alignment (MSA) of related proteins, whose co-evolving positions hint at which residues touch; its Evoformer (48 blocks of attention over the MSA and over residue pairs, with triangle updates) builds the pair picture; and its structure module places a full backbone frame per residue (invariant point attention) and the side chains, refined in several recycles. It was trained on ~170,000 structures from the Protein Data Bank and outputs a confidence (pLDDT). Here the "network" is faked: the distogram is the true distances plus noise that grows with distance ($\sigma_{ij} \propto d_{ij}$), and pairs beyond the last bin are only known to be far. The attention block is trained for a few hundred steps on this one map, so it memorises rather than predicts; it shows what attention over residue pairs looks like, and because its read-out puts each residue at a point in a 6-dimensional space it cannot fit independent noise, so it smooths it. The structure module is honest distance geometry. Distances cannot tell a structure from its mirror image: MDS and gradient descent return either enantiomer, so the right hand is picked by the sign of the Cα pseudo-dihedrals (about $+50^\circ$ in a right-handed α-helix of L-amino acids) and both RMSDs are printed. Several random starts are folded at once because some get stuck with one helix of each hand; the lowest stress wins. Try raising the noise until the fold breaks, or turning the refinement off.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/biophysics/mini_alphafold/main.py
"""Main loop: target fold → predicted distogram → (attention refinement) → structure module → alignment → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import classical_mds, distogram, distogram_mean_var, pair_distances, rmsd, tm_score

D_FAR = 18.0  # Å: beyond this the binned mean is biased (the distogram ends at 22 Å), so only say "far apart"


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    target = tensors.target_structure(params["helix_length"], dev)  # (N, 3) Å
    n = target.shape[0]
    D = pair_distances(target)  # (N, N) true distances
    print(f"target: helix–turn–helix, {n} residues (Cα only), radius of gyration {_rg(target):.1f} Å")

    # --- step 1: the network's job, a distogram ---
    d_noisy, sigma = tensors.noisy_distances(D, params["noise"])
    centres = tensors.bin_centres(dev)  # (K,)
    p = distogram(d_noisy, sigma, centres)  # (N, N, K)
    d_hat, var = distogram_mean_var(p, centres)  # (N, N), (N, N)
    w = 1.0 / (var + 0.1)  # confident pairs count more
    far = d_hat > D_FAR
    near = ~far & ~torch.eye(n, dtype=torch.bool, device=dev)
    print(f"distogram: {len(centres)} bins 2–22 Å; {far.sum().item() // 2} pairs only known as '≥ {D_FAR:.0f} Å'")
    print(f"  mean |d̂ - D| over near pairs: {(d_hat - D)[near].abs().mean():.2f} Å (noise level {params['noise']})")

    attn = None
    if params["refine"] == "on":
        feats = tensors.positional_features(n, 32, dev)  # (N, E)
        d_used, attn = solve.refine(d_hat, w, far, D_FAR, feats, 60 if fast else 400)
        print(f"  after attention refinement:  {(d_used - D)[near].abs().mean():.2f} Å")
    else:
        d_used = d_hat

    # --- step 2: the structure module ---
    x0 = solve.initial_guesses(d_used, params["starts"])  # (B, N, 3), start 0 = classical MDS
    mds = classical_mds(d_used)
    print(
        f"classical MDS alone: RMSD {rmsd(mds, target):.2f} Å, its mirror image {rmsd(solve.mirror(mds), target):.2f} Å"
    )
    n_steps = 80 if fast else params["steps"]
    at, snaps, history, final = solve.fold(x0, d_used, w, far, D_FAR, 10.0, n_steps)  # (T, B, N, 3)

    best = int(final.argmin())
    model, flipped = solve.choose_hand(snaps[-1, best])
    rms_all = solve.best_rmsd_any_hand(snaps[-1], target)  # (B,)
    print(f"gradient descent from {x0.shape[0]} starts ({n_steps} Adam steps): final stress {final.min():.1f} … ")
    print(f"  {final.max():.1f}; RMSD (best hand) from {rms_all.min():.2f} to {rms_all.max():.2f} Å")
    print(f"  lowest stress: start {best} ({'MDS' if best == 0 else 'random'}); mirror flipped: {flipped}")
    r_model, r_mirror = rmsd(model, target).item(), rmsd(solve.mirror(model), target).item()
    print(f"  chosen hand: RMSD {r_model:.2f} Å (the other enantiomer: {r_mirror:.2f} Å)")
    print(f"  TM-like score {tm_score(model, target):.3f}  (> 0.5 means the same fold)")

    # animate the best *random* start, so the fold visibly assembles from a cloud
    show = 1 + int(final[1:].argmin()) if x0.shape[0] > 1 else 0
    traj = snaps[:, show]
    if solve.choose_hand(traj[-1])[1]:
        traj = solve.mirror(traj)
    rms_traj = rmsd(traj, target)  # (T,)
    rms_starts = solve.best_rmsd_any_hand(snaps, target)  # (T, B)
    return plot.animate(target, traj, d_used, far, attn, rms_traj, rms_starts, at, out_dir / "mini_alphafold.gif")


def _rg(x: torch.Tensor) -> float:
    return torch.sqrt(((x - x.mean(0)) ** 2).sum(1).mean()).item()


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/biophysics/mini_alphafold/equation.py
r"""Physics → tensor translation: from a predicted distance map to 3D coordinates.

Indices: i, j residues (N);  a, b Cartesian axes (3);  k distance bins (K);  e feature channels (E).

1. Distogram (what the network predicts):  p_{ijk} = probability that residues i, j are c_k apart.
   Mean and spread:  d̂_ij = p_{ijk} c_k   (k summed),   Var_ij = p_{ijk} c_k² - d̂_ij².
2. Attention over residues (the core operation of a transformer):
   a_ij = q_{ie} k_{je} / √E + b_ij   (e summed),   A_ij = softmax_j(a_ij),   h'_ie = h_ie + A_ij v_je   (j summed).
3. Structure: find x_{ia} with r_ij = √((x_ia - x_ja)(x_ia - x_ja)) (a summed) close to d̂_ij, i.e. minimise the stress
   S = Σ_{i<j} w_ij (r_ij - d̂_ij)²  +  far pairs: Σ relu(d_far - r_ij)²  +  k_b Σ_i (r_{i,i+1} - 3.8 Å)².
4. Classical MDS: G_ij = -½ J_{ik} D²_{kl} J_{lj} (k, l summed) is the Gram matrix x_ia x_ja of centred coordinates.
5. Kabsch: H_ab = p_ia q_ib (i summed) = U S Vᵀ;  the best rotation is R = V diag(1, 1, det(V Uᵀ)) Uᵀ.
"""

import math

import torch

CA_CA = 3.8  # Å


def pair_distances(x: torch.Tensor) -> torch.Tensor:
    """r_ij = |x_i - x_j| for coordinates (..., N, 3) → (..., N, N).  The axis index a is summed."""
    diff = x[..., :, None, :] - x[..., None, :, :]  # (..., N, N, 3)
    return torch.sqrt(torch.einsum("...ija,...ija->...ij", diff, diff) + 1e-9)


def distogram(d: torch.Tensor, sigma: torch.Tensor, centres: torch.Tensor) -> torch.Tensor:
    """p_ijk = softmax_k(-(d_ij - c_k)² / 2σ_ij²): a predicted distance spread over K bins.   (N, N, K)

    Distances beyond the last bin pile up in it: the distogram only says "22 Å or more".
    """
    logits = -((d[..., None] - centres) ** 2) / (2 * sigma[..., None] ** 2)
    return torch.softmax(logits, dim=-1)


def distogram_mean_var(p: torch.Tensor, centres: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """d̂_ij = p_ijk c_k and Var_ij = p_ijk c_k² - d̂_ij²   (k summed).   Both (N, N)."""
    mean = torch.einsum("ijk,k->ij", p, centres)
    var = torch.einsum("ijk,k->ij", p, centres**2) - mean**2
    return mean, var


def attention(h: torch.Tensor, Wq, Wk, Wv, bias: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """One head of self-attention with a pair bias.  h: (N, E), W*: (E, E), bias: (N, N).

    a_ij = q_ie k_je / √E + b_ij    →   A_ij = softmax over j    →   h'_ie = h_ie + A_ij v_je.
    The pair bias b_ij is how AlphaFold's Evoformer lets the pair (distance) picture steer the residues.
    """
    q, k, v = h @ Wq, h @ Wk, h @ Wv  # (N, E) each
    logits = torch.einsum("ie,je->ij", q, k) / math.sqrt(h.shape[1]) + bias
    A = torch.softmax(logits, dim=1)  # each row sums to 1: residue i's attention over all j
    return h + torch.einsum("ij,je->ie", A, v), A


def stress(x: torch.Tensor, d: torch.Tensor, w: torch.Tensor, far: torch.Tensor, d_far: float, k_bond: float):
    """S per structure for coordinates x (B, N, 3); d, w, far (bool) are (N, N).   Returns (B,).

    Near pairs: weighted squared error of the distance.  Far pairs ("≥ d_far"): only penalised when too close.
    Bonds: consecutive Cα atoms must stay 3.8 Å apart.
    """
    r = pair_distances(x)  # (B, N, N)
    upper = torch.triu(torch.ones_like(w, dtype=torch.bool), diagonal=1)
    near = upper & ~far
    s_near = (w * (r - d) ** 2 * near).sum((1, 2))
    s_far = (torch.relu(d_far - r) ** 2 * (upper & far)).sum((1, 2))
    bonds = r.diagonal(offset=1, dim1=1, dim2=2)  # (B, N-1): r_{i,i+1}
    s_bond = k_bond * ((bonds - CA_CA) ** 2).sum(1)
    return s_near + s_far + s_bond


def classical_mds(D: torch.Tensor, dim: int = 3) -> torch.Tensor:
    """Coordinates (N, dim) whose distances best match D (N, N): G = -½ J D² J, then the top eigenvectors.

    For exact Euclidean distances G_ij = x_ia x_ja exactly, so x = V Λ^{1/2} recovers the structure up to a rotation,
    a translation and a mirror image (distances cannot tell left from right).
    """
    n = D.shape[0]
    J = torch.eye(n, dtype=D.dtype, device=D.device) - 1.0 / n  # centring matrix
    G = -0.5 * torch.einsum("ik,kl,lj->ij", J, D**2, J)
    evals, evecs = torch.linalg.eigh(G)  # ascending
    top = evals[-dim:].flip(0).clamp(min=0)
    return evecs[:, -dim:].flip(1) * torch.sqrt(top)


def kabsch(P: torch.Tensor, Q: torch.Tensor) -> torch.Tensor:
    """Rotate and translate P (..., N, 3) onto Q (N, 3) with the best proper rotation (det R = +1).   (..., N, 3)"""
    Pc = P - P.mean(-2, keepdim=True)
    Qc = Q - Q.mean(-2, keepdim=True)
    H = torch.einsum("...ia,...ib->...ab", Pc, Qc.expand_as(Pc))  # (..., 3, 3) covariance
    U, _, Vh = torch.linalg.svd(H)
    d = torch.sign(torch.linalg.det(Vh.mT @ U.mT))  # -1 would be a reflection: forbid it
    S = torch.diag_embed(torch.stack([torch.ones_like(d), torch.ones_like(d), d], -1))
    R = Vh.mT @ S @ U.mT  # (..., 3, 3)
    return torch.einsum("...ab,...ib->...ia", R, Pc) + Q.mean(-2, keepdim=True)


def rmsd(P: torch.Tensor, Q: torch.Tensor) -> torch.Tensor:
    """√(mean_i |p_i - q_i|²) after Kabsch superposition.   (...)"""
    diff = kabsch(P, Q) - Q
    return torch.sqrt(torch.einsum("...ia,...ia->...", diff, diff) / Q.shape[0])


def tm_score(P: torch.Tensor, Q: torch.Tensor) -> torch.Tensor:
    """TM-like score (1/N) Σ_i 1 / (1 + (|p_i - q_i| / d0)²) on the Kabsch superposition; 1 = identical.

    d0 = 1.24 (N - 15)^{1/3} - 1.8 Å is the TM-score length scale. The real TM-score also searches over
    superpositions to maximise the score, so this value is a lower bound.
    """
    n = Q.shape[0]
    d0 = max(1.24 * max(n - 15, 1) ** (1 / 3) - 1.8, 0.5)
    dist = torch.linalg.norm(kabsch(P, Q) - Q, dim=-1)
    return (1.0 / (1.0 + (dist / d0) ** 2)).mean(-1)


def pseudo_dihedrals(x: torch.Tensor) -> torch.Tensor:
    """Dihedral angle of every four consecutive Cα atoms, (..., N, 3) → (..., N-3), in degrees.

    About +50° in a right-handed α-helix; a mirror image flips every sign. That is how we pick the enantiomer.
    """
    b1, b2, b3 = (x[..., 1:, :] - x[..., :-1, :]).unfold(-2, 3, 1).unbind(-1)  # bond vectors i→i+1, ...
    n1, n2 = torch.cross(b1, b2, dim=-1), torch.cross(b2, b3, dim=-1)  # normals of the two planes
    y = torch.linalg.norm(b2, dim=-1) * (b1 * n2).sum(-1)
    return torch.rad2deg(torch.atan2(y, (n1 * n2).sum(-1)))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/biophysics/mini_alphafold/tensors.py
"""Build the concrete tensors: the target fold, the distogram bins, a noisy "network prediction", residue features.

Units: Å. Only the Cα atom of each residue is modelled (one point per amino acid), 3.8 Å apart along the chain.
"""

import math

import torch

CA_CA = 3.8  # Å, distance between consecutive Cα atoms (fixed by the planar peptide bond)
HELIX_RADIUS = 2.3  # Å, radius of the Cα spiral of an α-helix
HELIX_RISE = 1.5  # Å per residue along the helix axis
HELIX_TURN = math.radians(100.0)  # 3.6 residues per turn
HELIX_SEPARATION = 10.0  # Å between the two helix axes (typical for packed helices)
LOOP = 3  # residues in the turn between the helices


def target_structure(n_helix: int, device: torch.device) -> torch.Tensor:
    """Cα trace of a synthetic helix–turn–helix, shape (N, 3) with N = 2 n_helix + LOOP.

    Two right-handed α-helices run antiparallel, side by side, joined by a short loop. Everything is built
    geometrically, so the "known structure" is exact (no coordinates copied from the PDB).
    """
    t = torch.arange(n_helix, dtype=torch.float64)
    phi = HELIX_TURN * t
    # helix 1 goes up the z-axis at x = -s/2; right-handed because the angle grows with z
    h1 = torch.stack(
        [-HELIX_SEPARATION / 2 + HELIX_RADIUS * torch.cos(phi), HELIX_RADIUS * torch.sin(phi), HELIX_RISE * t], -1
    )
    # helix 2: helix 1 half a turn on, rotated by π about the x-axis (still right-handed), moved to x = +s/2
    top = HELIX_RISE * (n_helix - 1)
    h2 = torch.stack(
        [
            HELIX_SEPARATION / 2 - HELIX_RADIUS * torch.cos(phi),
            HELIX_RADIUS * torch.sin(phi),
            top - HELIX_RISE * t,
        ],
        -1,
    )
    loop = _arc(h1[-1], h2[0], LOOP)
    return torch.cat([h1, loop, h2]).float().to(device)


def _arc(p: torch.Tensor, q: torch.Tensor, m: int) -> torch.Tensor:
    """m points on a circular arc from p to q (exclusive), bulging upwards, with all m + 1 chords equal to 3.8 Å."""
    span = torch.linalg.norm(q - p).item()
    # find the arc angle Θ with  sin(Θ/2) / sin(Θ / (2(m+1))) = span / 3.8  by bisection
    lo, hi = 1e-6, 2 * math.pi - 1e-6
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        ratio = math.sin(mid / 2) / math.sin(mid / (2 * (m + 1)))
        lo, hi = (mid, hi) if ratio > span / CA_CA else (lo, mid)
    theta = 0.5 * (lo + hi)
    radius = span / (2 * math.sin(theta / 2))
    e = (q - p) / span  # along the chord
    up = torch.tensor([0.0, 0.0, 1.0], dtype=p.dtype)
    up = up - (up @ e) * e
    up = up / torch.linalg.norm(up)  # perpendicular to the chord, pointing up
    centre = 0.5 * (p + q) - radius * math.cos(theta / 2) * up
    a = -theta / 2 + theta * torch.arange(1, m + 1, dtype=p.dtype) / (m + 1)  # angles of the loop points
    return centre + radius * (torch.sin(a)[:, None] * e + torch.cos(a)[:, None] * up)


def bin_centres(device: torch.device, n_bins: int = 64, d_min: float = 2.0, d_max: float = 22.0) -> torch.Tensor:
    """c_k, shape (K,): 64 distance bins between 2 and 22 Å, as in AlphaFold's distogram head."""
    edges = torch.linspace(d_min, d_max, n_bins + 1, device=device)
    return 0.5 * (edges[1:] + edges[:-1])


def noisy_distances(D: torch.Tensor, noise: float, seed: int = 0) -> tuple[torch.Tensor, torch.Tensor]:
    """Stand-in for a trained network: d̃_ij = D_ij + σ_ij ε_ij, symmetric, with σ_ij = noise · D_ij / 10 Å.

    Close pairs are predicted sharply and far pairs vaguely, as real distograms are. Returns (d̃, σ), both (N, N).
    """
    gen = torch.Generator().manual_seed(seed)
    eps = torch.randn(D.shape, generator=gen).to(D.device)
    eps = (eps + eps.T) / math.sqrt(2.0)  # symmetric: d_ij and d_ji are the same prediction
    sigma = noise * D / 10.0 + 0.2  # (N, N); a 0.2 Å floor
    d = (D + sigma * eps).clamp(min=2.0)
    d.fill_diagonal_(0.0)
    return d, sigma


def positional_features(n: int, dim: int, device: torch.device) -> torch.Tensor:
    """Sinusoidal encoding of the residue index i, shape (N, E): the only "sequence" information the toy sees."""
    i = torch.arange(n, device=device, dtype=torch.float32)[:, None]
    freq = torch.exp(-math.log(100.0) * torch.arange(0, dim, 2, device=device) / dim)[None, :]
    return torch.cat([torch.sin(i * freq), torch.cos(i * freq)], dim=1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/biophysics/mini_alphafold/solve.py
"""The two stages: refine the distance map with a tiny attention block, then fold coordinates to fit it."""

import torch

from problems._common.progress import steps

from .equation import attention, classical_mds, pair_distances, pseudo_dihedrals, rmsd, stress

READOUT_DIM = 6  # dimension of the feature space the read-out maps each residue into


def refine(d: torch.Tensor, w: torch.Tensor, far: torch.Tensor, d_far: float, feats: torch.Tensor, n_steps: int):
    """Train one attention block for n_steps to reproduce the predicted distances d (N, N); returns (d_ref, A).

    Each residue gets an embedding h_i = (positional features + a learned vector); after attention, a linear
    read-out puts it at a point u_i in a 6-dimensional feature space and the refined distance is |u_i - u_j|.
    N points in 6D cannot fit N² independent noise values, so the fit smooths the noise out.
    """
    torch.manual_seed(0)
    n, e = feats.shape
    p = {
        "h": torch.zeros(n, e),
        "Wq": torch.randn(e, e) / e**0.5,
        "Wk": torch.randn(e, e) / e**0.5,
        "Wv": torch.randn(e, e) / e**0.5,
        "Wout": torch.randn(e, READOUT_DIM) * 3.0,
        "tau": torch.tensor(4.0),
    }
    p = {k: v.to(d.device).requires_grad_() for k, v in p.items()}
    opt = torch.optim.Adam(p.values(), lr=0.02)
    for _ in steps(n_steps, "attention training"):
        d_ref, A = _forward(p, feats, d)
        loss = stress_2d(d_ref, d, w, far, d_far)
        opt.zero_grad()
        loss.backward()
        opt.step()
    with torch.no_grad():
        d_ref, A = _forward(p, feats, d)
    return d_ref, A


def _forward(p: dict, feats: torch.Tensor, d: torch.Tensor):
    h = feats + p["h"]  # (N, E)
    bias = -d / p["tau"].abs().clamp(min=0.5)  # pair bias: residues predicted to be close attend to each other
    h2, A = attention(h, p["Wq"], p["Wk"], p["Wv"], bias)
    u = h2 @ p["Wout"]  # (N, 6)
    return pair_distances(u), A


def stress_2d(r: torch.Tensor, d: torch.Tensor, w: torch.Tensor, far: torch.Tensor, d_far: float) -> torch.Tensor:
    """Same fit as the structure stress, written on a distance map r (N, N) instead of coordinates."""
    upper = torch.triu(torch.ones_like(far), diagonal=1)
    return (w * (r - d) ** 2 * (upper & ~far)).sum() + (torch.relu(d_far - r) ** 2 * (upper & far)).sum()


def initial_guesses(d: torch.Tensor, n_random: int, seed: int = 0) -> torch.Tensor:
    """Starts for gradient descent, shape (B, N, 3): classical MDS first, then random clouds a few Å across."""
    gen = torch.Generator().manual_seed(seed)
    n = d.shape[0]
    random = 6.0 * torch.randn(n_random, n, 3, generator=gen).to(d.device)
    return torch.cat([classical_mds(d)[None], random])


def fold(x0: torch.Tensor, d, w, far, d_far: float, k_bond: float, n_steps: int, n_snap: int = 90):
    """Gradient descent (Adam) on the stress for every start at once.

    Returns (snapshot steps (T,), snapshots (T, B, N, 3), stress history (n_steps, B), final stress (B,)).
    Snapshots are spaced quadratically in the step number: most of the folding happens early.
    """
    x = x0.clone().requires_grad_()
    opt = torch.optim.Adam([x], lr=0.5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n_steps, eta_min=0.01)
    keep = set((n_steps * (torch.arange(1, n_snap + 1) / n_snap) ** 2).round().long().tolist())
    snaps, at, history = [x0.clone()], [0], []
    for step in steps(n_steps, "structure module"):
        S = stress(x, d, w, far, d_far, k_bond)  # (B,)
        opt.zero_grad()
        S.sum().backward()  # the starts are independent: one backward pass serves them all
        opt.step()
        sched.step()
        history.append(S.detach())
        if step + 1 in keep:
            snaps.append(x.detach().clone())
            at.append(step + 1)
    with torch.no_grad():
        final = stress(x, d, w, far, d_far, k_bond)
    return torch.tensor(at), torch.stack(snaps), torch.stack(history), final


def mirror(x: torch.Tensor) -> torch.Tensor:
    """The mirror image (enantiomer): flip the z coordinate.   (..., N, 3)"""
    return x * torch.tensor([1.0, 1.0, -1.0], device=x.device)


def choose_hand(x: torch.Tensor) -> tuple[torch.Tensor, bool]:
    """Distances cannot tell a structure from its mirror image, so pick the one whose helices are right-handed
    (mean Cα pseudo-dihedral > 0, as for L-amino acids). Returns (structure, whether it was flipped)."""
    flip = pseudo_dihedrals(x).median().item() < 0
    return (mirror(x) if flip else x), flip


def best_rmsd_any_hand(x: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """min(RMSD(x), RMSD(mirror x)): how close the shape is, ignoring the hand.   (...)"""
    return torch.minimum(rmsd(x, target), rmsd(mirror(x), target))


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/biophysics/mini_alphafold/plot.py
"""Animate the structure module: the chain folding into place (3D, rotating) next to the distance maps."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .equation import kabsch, pair_distances


def animate(
    target: torch.Tensor,
    traj: torch.Tensor,
    d_used: torch.Tensor,
    far: torch.Tensor,
    attn: torch.Tensor | None,
    rms_traj: torch.Tensor,
    rms_starts: torch.Tensor,
    at: torch.Tensor,
    path: Path,
) -> Path:
    """target (N, 3); traj (T, N, 3) snapshots of one start, taken at steps at (T,); d_used, far (N, N);
    attn (N, N) or None; RMSD of the shown start (T,) and of every start (T, B)."""
    tgt = target.cpu().numpy()
    aligned = kabsch(traj, target).cpu().numpy()  # (T, N, 3) every snapshot superposed on the target
    maps = pair_distances(traj).cpu().numpy()  # (T, N, N)
    pred = np.where(far.cpu().numpy(), 22.0, d_used.cpu().numpy())
    rt, rs = rms_traj.cpu().numpy(), rms_starts.cpu().numpy()
    T, N = aligned.shape[:2]
    it = at.cpu().numpy()
    n_steps = int(it[-1])
    cmap = plt.get_cmap("viridis", 11)  # discrete colours keep the GIF small

    fig = plt.figure(figsize=(10.5, 4.8))
    grid = fig.add_gridspec(2, 3, width_ratios=[1.5, 1, 1])
    ax3 = fig.add_subplot(grid[:, 0], projection="3d")
    ax_p = fig.add_subplot(grid[0, 1])
    ax_r = fig.add_subplot(grid[0, 2])
    ax_a = fig.add_subplot(grid[1, 1])
    ax_c = fig.add_subplot(grid[1, 2])

    centre = tgt.mean(0)
    half = np.abs(tgt - centre).max() + 1
    ax3.plot(*tgt.T, color="#9ca3af", lw=1.2, alpha=0.8, label="target")
    (chain,) = ax3.plot([], [], [], color="#374151", lw=1.5)
    beads = ax3.scatter(*aligned[0].T, c=np.arange(N), cmap=plt.get_cmap("rainbow", 8), s=22, depthshade=False)
    for a, setter in zip(range(3), (ax3.set_xlim, ax3.set_ylim, ax3.set_zlim)):
        setter(centre[a] - half, centre[a] + half)
    ax3.set_box_aspect((1, 1, 1), zoom=1.5)
    ax3.set_axis_off()  # no panes or ticks: fewer colours and faster frames; the rotation shows the depth
    ax3.legend(loc="upper left", fontsize=8)
    title3 = ax3.set_title("")

    for ax, _, title in (
        (ax_p, pred, r"predicted $\hat d_{ij}$ (Å)"),
        (ax_r, maps[0], r"model distances $r_{ij}$ (Å)"),
    ):
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        ax.set_title(title, fontsize=9)
    ax_p.imshow(pred, cmap=cmap, vmin=0, vmax=22)
    im_r = ax_r.imshow(np.minimum(maps[0], 22), cmap=cmap, vmin=0, vmax=22)
    fig.colorbar(im_r, ax=ax_r, fraction=0.046, pad=0.04)

    ax_a.set_xticks([])
    ax_a.set_yticks([])
    ax_a.grid(False)
    if attn is not None:
        ax_a.imshow(attn.cpu().numpy(), cmap=plt.get_cmap("Blues", 9))
        ax_a.set_title("attention $A_{ij}$ (row i → j)", fontsize=9)
    else:
        ax_a.text(0.5, 0.5, "refinement off", ha="center", va="center", transform=ax_a.transAxes)

    for b in range(rs.shape[1]):
        ax_c.plot(it, rs[:, b], color="#d1d5db", lw=0.8)
    ax_c.plot(it, rt, color="#93c5fd", lw=1)
    (trace,) = ax_c.plot([], [], color="#2563eb", lw=1.8)
    (dot,) = ax_c.plot([], [], "o", color="#2563eb", ms=4)
    ax_c.set_yscale("log")
    ax_c.set_yticks([0.5, 1, 2, 5, 10], ["0.5", "1", "2", "5", "10"])  # plain labels: no mathtext per frame
    ax_c.minorticks_off()
    ax_c.set_xlim(0, n_steps)
    ax_c.set_xlabel("gradient step", fontsize=8, labelpad=1)
    ax_c.set_ylabel("RMSD (Å)", fontsize=8)
    ax_c.set_title("all starts (grey), shown (blue)", fontsize=9)
    ax_c.tick_params(labelsize=7)

    def update(t: int):
        x = aligned[t]
        chain.set_data_3d(x[:, 0], x[:, 1], x[:, 2])
        beads._offsets3d = (x[:, 0], x[:, 1], x[:, 2])
        ax3.view_init(elev=18, azim=-60 + 200 * t / max(T - 1, 1))
        im_r.set_data(np.minimum(maps[t], 22))
        trace.set_data(it[: t + 1], rt[: t + 1])
        dot.set_data(it[t : t + 1], rt[t : t + 1])
        title3.set_text(f"step {it[t]:.0f}: RMSD {rt[t]:.2f} Å")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(T, 72), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'helix_length': 16,
    'noise': 0.5,
    'refine': 'on',
    'starts': 16,
    'steps': 800
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.biophysics.mini_alphafold")]:
    del sys.modules[name]
main = importlib.import_module("problems.biophysics.mini_alphafold.main")
problem_dir = Path("problems/biophysics/mini_alphafold")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
